In [1]:
import wikipediaapi
import re
import time

# Инициализация с правильным User-Agent
wiki = wikipediaapi.Wikipedia(
    user_agent='BerryEmbeddingLab/1.0 (study-project)',
    language='ru',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

berry_names = [
    "Малина", "Клубника", "Черника", "Земляника", "Голубика", 
    "Ежевика", "Крыжовник", "Смородина", "Клюква", "Облепиха"
]

corpus = []
for berry in berry_names:
    try:
        page = wiki.page(berry)
        
        if page.exists():
            text = page.text.lower()
            # Оставляем только кириллицу
            text = re.sub(r'[^а-яё\s]', ' ', text)
            words = [w for w in text.split() if len(w) > 1]
            corpus.append(words)
            print(f"Собрана статья: {berry} ({len(words)} слов)")
        else:
            print(f"Статья {berry} не найдена!")
            
    except Exception as e:
        print(f"Ошибка с {berry}: {e}")
        
    # КЛЮЧЕВОЙ МОМЕНТ: Спим 2 секунды перед следующим запросом
    time.sleep(2)

print(f"\nИтого собрано документов: {len(corpus)}")

# corpus — это список списков слов: [["малина", "обыкновенная", "полукустарник", ...], [...]]

Собрана статья: Малина (821 слов)
Собрана статья: Клубника (80 слов)
Собрана статья: Черника (1261 слов)
Собрана статья: Земляника (972 слов)
Собрана статья: Голубика (594 слов)
Собрана статья: Ежевика (545 слов)
Собрана статья: Крыжовник (734 слов)
Собрана статья: Смородина (493 слов)
Собрана статья: Клюква (1647 слов)
Собрана статья: Облепиха (337 слов)

Итого собрано документов: 10


In [2]:
import json
import os

# Создаем папку data, если её нет (хорошая практика для будущей структуры проекта)
os.makedirs("data", exist_ok=True)
corpus_path = "data/berry_corpus.json"

# 1. Сохраняем корпус в файл
with open(corpus_path, "w", encoding="utf-8") as f:
    # ensure_ascii=False сохранит кириллицу в читаемом виде, а не в \uXXXX
    json.dump(corpus, f, ensure_ascii=False, indent=2)
    
print(f"Корпус успешно сохранён в '{corpus_path}'!")

Корпус успешно сохранён в 'data/berry_corpus.json'!


In [3]:
corpus_path = "data/berry_corpus.json"

# 2. Пример того, как загружать данные (можешь использовать в следующих сессиях)
with open(corpus_path, "r", encoding="utf-8") as f:
    loaded_corpus = json.load(f)

# Немного статистики для проверки
total_words = sum(len(doc) for doc in loaded_corpus)
vocab_size = len(set(word for doc in loaded_corpus for word in doc))

print(f"\nСтатистика загруженного корпуса:")
print(f"Документов: {len(loaded_corpus)}")
print(f"Всего слов: {total_words}")
print(f"Уникальных слов (размер словаря): {vocab_size}")


Статистика загруженного корпуса:
Документов: 10
Всего слов: 7484
Уникальных слов (размер словаря): 3565


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# Собираем словарь
all_words = [word for doc in corpus for word in doc]
vocab = list(set(all_words))
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
EMBED_DIM = 50
CONTEXT_SIZE = 2  # 2 слова слева, 2 слова справа

# Генерируем датасет
data = []
for doc in corpus:
    for i in range(CONTEXT_SIZE, len(doc) - CONTEXT_SIZE):
        context = (
            [doc[i - j - 1] for j in range(CONTEXT_SIZE)] + 
            [doc[i + j + 1] for j in range(CONTEXT_SIZE)]
        )
        target = doc[i]
        data.append((context, target))

# Самая простая архитектура
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, inputs):
        # inputs - это тензор индексов слов контекста
        embeds = self.embeddings(inputs) # размер: (4, 50)
        # Усредняем эмбеддинги контекста (получаем вектор 1x50)
        hidden = embeds.mean(dim=0).unsqueeze(0) 
        out = self.linear(hidden) # размер: (1, vocab_size)
        return out

model = CBOW(VOCAB_SIZE, EMBED_DIM)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Учебный цикл (MVP вариант без батчей для простоты)
print("Начинаем обучение CBOW...")
for epoch in range(5):
    total_loss = 0
    for context, target in data:
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)
        model.zero_grad()
        
        log_probs = model(context_idxs)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

# Как достать эмбеддинг после обучения:
word_idx = word_to_ix["малина"]
berry_embedding = model.embeddings(torch.tensor(word_idx))

Начинаем обучение CBOW...
Epoch 1, Loss: 60770.03
Epoch 2, Loss: 58837.57
Epoch 3, Loss: 57007.24
Epoch 4, Loss: 55328.07
Epoch 5, Loss: 53801.80


In [5]:
from gensim.models import FastText

print("Обучаем FastText...")
# Обучение в одну строчку
ft_model = FastText(sentences=corpus, vector_size=50, window=3, min_count=1, epochs=10)

# Проверяем качество эмбеддингов
word = "черника"
print(f"\nСамые похожие слова на '{word}':")
for similar_word, similarity in ft_model.wv.most_similar(word, topn=5):
    print(f" - {similar_word}: {similarity:.3f}")

# Демонстрация главной фичи FastText (OOV слова)
# Слова "голубичка" или "малинка" скорее всего нет в обучающей выборке
oov_word = "ежевичка" 
if oov_word not in ft_model.wv.key_to_index:
    print(f"\nСлова '{oov_word}' не было в корпусе!")
    # Но мы всё равно можем получить его вектор:
    oov_vector = ft_model.wv[oov_word]
    print(f"Размерность полученного вектора: {oov_vector.shape}")
    
    # И даже найти для него соседей
    print(f"Ближайшие к '{oov_word}':", ft_model.wv.most_similar(oov_word, topn=3))

Обучаем FastText...

Самые похожие слова на 'черника':
 - черники: 0.999
 - черникой: 0.999
 - чернику: 0.999
 - чернике: 0.999
 - земляника: 0.999

Слова 'ежевичка' не было в корпусе!
Размерность полученного вектора: (50,)
Ближайшие к 'ежевичка': [('простые', 0.9938364624977112), ('ежевикой', 0.9933922290802002), ('ежевики', 0.9933831095695496)]


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# === 1. Подготовка данных и DataLoader ===
# Предполагаем, что loaded_corpus у нас уже есть из предыдущей ячейки
all_words = [word for doc in loaded_corpus for word in doc]
vocab = list(set(all_words))
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
EMBED_DIM = 64
CONTEXT_SIZE = 2  # 2 слева, 2 справа = 4 слова в контексте
BATCH_SIZE = 128

class BerryDataset(Dataset):
    def __init__(self, corpus, context_size, word_to_ix):
        self.data = []
        for doc in corpus:
            for i in range(context_size, len(doc) - context_size):
                context = (
                    [doc[i - j - 1] for j in range(context_size)] + 
                    [doc[i + j + 1] for j in range(context_size)]
                )
                target = doc[i]
                # Сохраняем индексы
                self.data.append((
                    [word_to_ix[w] for w in context],
                    word_to_ix[target]
                ))
                
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

dataset = BerryDataset(loaded_corpus, CONTEXT_SIZE, word_to_ix)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# === 2. Архитектуры моделей ===

class SimpleCBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, context):
        # context: (batch, seq_len)
        embeds = self.embeddings(context)  # (batch, seq_len, embed_dim)
        # Простое усреднение контекста
        hidden = embeds.mean(dim=1)        # (batch, embed_dim)
        return self.linear(hidden)

class AttentionCBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        # Тот самый шаг вперед: слой внимания
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, context):
        embeds = self.embeddings(context)  # (batch, seq_len, embed_dim)
        # Self-Attention: контекст смотрит сам на себя
        attn_out, _ = self.attention(embeds, embeds, embeds) # (batch, seq_len, embed_dim)
        # Усредняем уже взвешенные вниманием векторы
        hidden = attn_out.mean(dim=1)
        return self.linear(hidden)

# === 3. Цикл обучения ===

def train_model(model, name, epochs=15):
    print(f"\n--- Обучение {name} ---")
    # Используем твою RTX 5070 Ti, если доступна
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    for epoch in range(epochs):
        total_loss = 0
        for context, target in dataloader:
            context, target = context.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(context)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")
            
    return model.cpu()

model_simple = train_model(SimpleCBOW(VOCAB_SIZE, EMBED_DIM), "SimpleCBOW")
model_attn = train_model(AttentionCBOW(VOCAB_SIZE, EMBED_DIM), "AttentionCBOW")

# === 4. Сравнение эмбеддингов ===

def get_similar_words(model, word, top_n=5):
    if word not in word_to_ix:
        return ["Слово не в словаре!"]
    
    # Достаем матрицу весов (наши эмбеддинги)
    embeddings = model.embeddings.weight.detach()
    word_idx = word_to_ix[word]
    word_vec = embeddings[word_idx].unsqueeze(0)
    
    # Считаем косинусное сходство со всеми словами
    similarities = F.cosine_similarity(word_vec, embeddings)
    
    # Берем top_n + 1 (так как самое похожее слово — это само слово)
    top_indices = torch.topk(similarities, top_n + 1).indices.tolist()
    
    results = []
    for idx in top_indices:
        if idx != word_idx:
            results.append((ix_to_word[idx], similarities[idx].item()))
    return results

test_word = "малина"
print(f"\n=== Ближайшие слова к '{test_word}' ===")

print("\nSimpleCBOW (Усреднение):")
for w, sim in get_similar_words(model_simple, test_word):
    print(f"  {w}: {sim:.3f}")

print("\nAttentionCBOW (Сложное взвешивание):")
for w, sim in get_similar_words(model_attn, test_word):
    print(f"  {w}: {sim:.3f}")


--- Обучение SimpleCBOW ---
Epoch 5/15, Loss: 4.3357
Epoch 10/15, Loss: 0.9982
Epoch 15/15, Loss: 0.2794

--- Обучение AttentionCBOW ---
Epoch 5/15, Loss: 1.5074
Epoch 10/15, Loss: 0.0770
Epoch 15/15, Loss: 0.0287

=== Ближайшие слова к 'малина' ===

SimpleCBOW (Усреднение):
  костяника: 0.426
  основании: 0.419
  болотах: 0.411
  журавика: 0.389
  используется: 0.358

AttentionCBOW (Сложное взвешивание):
  женские: 0.438
  вещество: 0.421
  напитки: 0.387
  жёлтые: 0.383
  мочевыводящих: 0.373


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import math
import random

# === 1. Обновляем словарь (добавляем спец-токены) ===
all_words = [word for doc in loaded_corpus for word in doc]
vocab = ["<PAD>", "<MASK>"] + list(set(all_words))
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
EMBED_DIM = 64
SEQ_LEN = 16  # Длина фрагмента текста
BATCH_SIZE = 128
MASK_PROB = 0.15  # Вероятность замаскировать слово

PAD_IDX = word_to_ix["<PAD>"]
MASK_IDX = word_to_ix["<MASK>"]

# === 2. Датасет для Masked Language Modeling ===
class MLMDataset(Dataset):
    def __init__(self, corpus, seq_len, word_to_ix):
        self.data = []
        # Нарезаем весь корпус на чанки длины seq_len
        flat_corpus = [word_to_ix[w] for doc in corpus for w in doc]
        for i in range(0, len(flat_corpus) - seq_len, seq_len):
            chunk = flat_corpus[i:i + seq_len]
            self.data.append(chunk)
                
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        chunk = self.data[idx]
        inputs = chunk.copy()
        targets = [-100] * SEQ_LEN  # -100 игнорируется в CrossEntropyLoss
        
        # Случайно маскируем 15% токенов
        for i in range(SEQ_LEN):
            if random.random() < MASK_PROB:
                inputs[i] = MASK_IDX
                targets[i] = chunk[i] # Модель должна угадать это слово
                
        return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

dataset_mlm = MLMDataset(loaded_corpus, SEQ_LEN, word_to_ix)
dataloader_mlm = DataLoader(dataset_mlm, batch_size=BATCH_SIZE, shuffle=True)

# === 3. Архитектура Mini-BERT ===
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MiniBERT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads=4, num_layers=2):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)
        
        # Полноценный блок Трансформера (Encoder)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim * 4, 
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Голова для предсказания замаскированного слова
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, src):
        # src: (batch, seq_len)
        padding_mask = (src == PAD_IDX) # Чтобы не обращать внимание на паддинги
        
        x = self.embeddings(src) * math.sqrt(EMBED_DIM)
        x = self.pos_encoder(x)
        
        # Пропускаем через слои трансформера
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        
        return self.linear(x) # (batch, seq_len, vocab_size)

# === 4. Обучение ===
print(f"\n--- Обучение MiniBERT ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert = MiniBERT(VOCAB_SIZE, EMBED_DIM).to(device)

# ignore_index=-100 означает, что лосс считается ТОЛЬКО для замаскированных слов
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.AdamW(model_bert.parameters(), lr=1e-3)

epochs = 30
for epoch in range(epochs):
    total_loss = 0
    for inputs, targets in dataloader_mlm:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        output = model_bert(inputs) # (batch, seq_len, vocab_size)
        
        # Меняем размерности для CrossEntropyLoss: (batch * seq_len, vocab_size)
        output = output.view(-1, VOCAB_SIZE)
        targets = targets.view(-1)
        
        loss = criterion(output, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader_mlm):.4f}")

model_bert = model_bert.cpu()

# === 5. Проверяем векторы ===
print("\nMiniBERT (Трансформер с маскированием):")
for w, sim in get_similar_words(model_bert, "малина"):
    print(f"  {w}: {sim:.3f}")


--- Обучение MiniBERT ---
Epoch 5/30, Loss: 8.0202
Epoch 10/30, Loss: 7.8091
Epoch 15/30, Loss: 7.6951
Epoch 20/30, Loss: 7.5870
Epoch 25/30, Loss: 7.4105
Epoch 30/30, Loss: 7.3352

MiniBERT (Трансформер с маскированием):
  меди: 0.412
  листьям: 0.399
  непал: 0.386
  одичавшем: 0.386
  поверхности: 0.371
